In [0]:
# =====================================================
# 1️⃣ Widgets & Fail-Fast Validation
# =====================================================
# REQUIRED PARAMETERS (supplied by the Master job Email_Notification task or manual runs):
#   run_id          : numeric run identifier
#   sender_email    : Gmail account used to send (must match the gmail_app_password)
#   recipient_email : notification recipient
#   catalog_name    : Unity Catalog name (e.g. 'banking') — see config/*.json

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("sender_email", "")
dbutils.widgets.text("recipient_email", "")
dbutils.widgets.text("catalog_name", "")

job_run_id = dbutils.widgets.get("run_id").strip()
sender_email = dbutils.widgets.get("sender_email").strip()
to_email = dbutils.widgets.get("recipient_email").strip()
catalog_name = dbutils.widgets.get("catalog_name").strip()

if not job_run_id:
    raise ValueError("run_id is required")
if not sender_email:
    raise ValueError("sender_email is required (supply via job parameter — see config/*.json)")
if not to_email:
    raise ValueError("recipient_email is required (supply via job parameter — see config/*.json)")
if not catalog_name:
    raise ValueError("catalog_name is required (e.g. 'banking' — see config/*.json)")

print("Job Run ID:", job_run_id)

In [0]:
audit_df = spark.sql(f"""
SELECT
    p.table_id AS Table_ID,
    t.table_name AS Table_Name,
    p.layer AS Layer,
    p.start_time AS Start_Time,
    p.end_time AS End_Time,
    p.status AS Status,
    p.number_of_records AS Number_Of_Records,
    p.error_message AS Error_Message
FROM {catalog_name}.metadata.pipeline_runs p
LEFT JOIN {catalog_name}.metadata.tables t
    ON p.table_id = t.table_id
WHERE p.run_id = {job_run_id}
ORDER BY p.table_id
""")

display(audit_df)

In [0]:
from pyspark.sql.functions import col

total = audit_df.count()
failed = audit_df.filter(col("status") != "SUCCESS").count()

if failed > 0:
    overall_status = "FAILED"
else:
    overall_status = "SUCCESS"

print("Total Tables:", total)
print("Failed Tables:", failed)
print("Overall Status:", overall_status)

In [0]:
pdf = audit_df.toPandas()

html_table = pdf.to_html(
    index=False,
    border=1,
    justify="center",
    classes="table table-striped"
)

print(html_table)

In [0]:
body = f"""
<html>
<body>

<p><b>This is an auto-generated email. Please do not reply.</b></p>

<p>
End-to-end run for <b>Source → Bronze → Silver → Gold</b> layers has completed.
Below is the execution summary:
</p>

<br>

{html_table}

<br>

<p>
For any queries please reach out to:
<b>{to_email}</b>
</p>

<br>

<p>
Thanks<br>
<b>Banking Support Team</b>
</p>

</body>
</html>
"""

print(body)

In [0]:
subject = f"NeoBank | {overall_status} | End to End Run"

print(subject)

In [0]:
gmail_app_password = dbutils.secrets.get(
    scope="banking-scope",
    key="gmail_app_password"
)

print("Gmail App Password retrieved successfully!")

In [0]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# Gmail account (parameterized — from widgets/job parameters)
EMAIL = sender_email

# Retrieve Gmail App Password from Databricks
APP_PASSWORD = gmail_app_password

# Receiver (already set from the recipient_email widget in cell 1)

# Create message
msg = MIMEMultipart()
msg["From"] = EMAIL
msg["To"] = to_email
msg["Subject"] = subject

msg.attach(MIMEText(body, "html"))

try:
    with smtplib.SMTP("smtp.gmail.com", 587) as server:
        server.starttls()
        server.login(EMAIL, APP_PASSWORD)
        server.send_message(msg)

    print("Email sent successfully!")

except Exception as e:
    print("Error sending email:", e)